## Download the dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mikhailklemin/kinopoisks-movies-reviews")

print("Path to dataset files:", path)

c:\Users\Asus\Desktop\LegendProjects\KinopoiskClassifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 222M/222M [00:39<00:00, 5.88MB/s] 

Extracting files...


Path to dataset files: C:\Users\Asus\.cache\kagglehub\datasets\mikhailklemin\kinopoisks-movies-reviews\versions\1


In [3]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

base = Path(path) / "dataset"
label_map = {"neg": 0, "neu": 1, "pos": 2}

def read_one(args):
    fp, cls = args
    text = fp.read_text(encoding="utf-8", errors="replace").strip()
    return {"text": text, "label": cls, "label_id": label_map[cls]}

tasks = [(fp, cls) for cls in label_map for fp in (base / cls).glob("*.txt")]

with ThreadPoolExecutor(max_workers=16) as ex:
    rows = list(ex.map(read_one, tasks))

df = pd.DataFrame(rows)

In [4]:
df.head()

,text,label,label_id
0,В 2003-ем году под руководством малоизвестного...,neg,0
1,"Грустно и печально. Грустно от того, что довол...",neg,0
2,Давным-давно Кира Найтли ворвалась на экран от...,neg,0
3,"Я, в общем, ничего против уравновешенного феми...",neg,0
4,"Измена — один из сюжетов, который всегда будет...",neg,0


## Чистка

In [5]:
before = len(df)
df = df[df["text"].str.len() > 0]              # пустые
df = df.drop_duplicates(subset="text")         # дубли текста
df = df.reset_index(drop=True)
print(f"Убрано {before - len(df)} пустых/дублей, осталось {len(df)}")
print(df["label"].value_counts())

Убрано 96 пустых/дублей, осталось 131573
label
pos    87091
neu    24678
neg    19804
Name: count, dtype: int64


## распределение длин текста в токенах RuBERT
### смотришь длины → выбираешь max_length → это число дальше зажимает batch size, скорость обучения, память, эффективность инференса и уходит в артефакт модели.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

#У DeepPavlov/rubert-base-cased максимум 512 токенов 
# длина в токенах для каждого текста
# add_special_tokens=True -> учитывает [CLS] и [SEP], которые реально добавятся при обучении
lengths = df["text"].apply(
    lambda t: len(tokenizer.encode(t, add_special_tokens=True, truncation=False))
)

print(lengths.describe(percentiles=[.5, .75, .9, .95, .99]))
print("макс:", lengths.max())
print("доля > 512 токенов:", (lengths > 512).mean().round(3))
print("доля > 256 токенов:", (lengths > 256).mean().round(3))

count    131573.000000
mean        483.449302
std         289.443068
min          12.000000
50%         415.000000
75%         619.000000
90%         887.000000
95%        1085.000000
99%        1408.000000
max        3100.000000
Name: text, dtype: float64
макс: 3100
доля > 512 токенов: 0.362
доля > 256 токенов: 0.783


### Вывод такой - много отзывов больше 512. Если отзыв не влезает в 512 токенов, берем начало и конец отзыва, середину выкидываем.

In [ ]:
def encode_head_tail(text, tokenizer, max_length=512, head=256, tail=254):
    # head + tail + [CLS] + [SEP] = 256 + 254 + 2 = 512
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    if len(ids) <= max_length - 2:
        kept = ids                                   # влезает целиком
    else:
        kept = ids[:head] + ids[-tail:]              # начало + конец, середина выкинута
    return tokenizer.prepare_for_model(kept, max_length=max_length, truncation=True)

## Стратифицированный сплит 70% train / 15% val / 15% test

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=42,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=42,
) 

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(part)} | доли классов:\n{part['label'].value_counts(normalize=True).round(3).to_dict()}")

train: 92101 | доли классов:
{'pos': 0.662, 'neu': 0.188, 'neg': 0.151}
val: 19736 | доли классов:
{'pos': 0.662, 'neu': 0.188, 'neg': 0.15}
test: 19736 | доли классов:
{'pos': 0.662, 'neu': 0.188, 'neg': 0.151}


## Артефакт label_map.json

In [7]:
import json

Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)